## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog. 

<b>Caution</b> ... This notebook requires significant execution time as there are 9319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer and typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and use of cloud computing services. 

### Load Python Dependencies

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

<h3>Extracting Landsat Data Using API Calls</h3> <p align="justify"> The API-based method allows us to efficiently access <b>Landsat</b> data for specific coordinates and time periods, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can query individual bands or compute indices like <b>NDMI</b> on-the-fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis. </p>

<p>The <b>compute_Landsat_values</b> function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:</p>

<ul>
  <li>A bounding box (bbox) is created around the latitude and longitude coordinates.</li>
  <li>The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.</li>
  <li>The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (<b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>) are loaded.</li>
  <li>Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.</li>
</ul>

<p><b>Why the buffer value is 0.00089831:</b></p>

<p>We want a ~100 m buffer around each point. At the equator, 1 degree ≈ 110 km. Therefore, the degree equivalent of 100 m is:</p>

<p style="text-align:center;">
  <em>buffer_deg = 100 m / 110,000 m/deg ≈ 0.00089831</em>
</p>

<p>This slightly adjusted value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.</p>


In [12]:
# Setup
tqdm.pandas()

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Wider search range, we'll filter to nearest date later
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # Load required bands
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # Compute medians
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)

        # Replace 0 with NaN
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
    
    except Exception as e:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })


In [3]:
def _empty_landsat_row():  # Changed
    return pd.Series({
        "blue": np.nan, "red": np.nan, "green": np.nan,
        "nir": np.nan, "swir16": np.nan, "swir22": np.nan
    }) 

In [4]:
# Changed: load baseline landsat (given by organizers)
base_ls = pd.read_csv("./data/landsat_features_training.csv")  # Changed

# Changed: unify keys (keep original string too)
KEYS = ["Latitude", "Longitude", "Sample Date"]  # Changed
base_ls["__date"] = pd.to_datetime(base_ls["Sample Date"], dayfirst=True, errors="coerce")  # Changed
base_ls["__key_date"] = base_ls["__date"].dt.strftime("%Y-%m-%d")  # Changed

# Changed: extract ONLY extra bands (blue/red), keep existing baseline bands untouched

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)  # Changed

# Changed: retry helper (handles API timeout)
import time

def retry(fn, tries=5, sleep=2):
    last_err = None
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            last_err = e
            time.sleep(sleep * (i + 1))
    raise last_err


def compute_Landsat_extra(row):
    lat = row["Latitude"]
    lon = row["Longitude"]
    date = pd.to_datetime(row["Sample Date"], dayfirst=True, errors="coerce")

    # Changed: invalid date guard
    if pd.isna(date):
        return pd.Series({"ls_blue": np.nan, "ls_red": np.nan})

    # Changed: smaller bbox reduces timeout probability
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2,
    ]

    try:
        # Changed: search with retry
        search = retry(lambda: catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime="2011-01-01/2015-12-31",
            query={"eo:cloud_cover": {"lt": 10}},
        ))

        items = search.item_collection()

        if not items:
            return pd.Series({"ls_blue": np.nan, "ls_red": np.nan})

        # Changed: convert sample date
        sample_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Changed: pick scene closest to sample date
        items = sorted(
            items,
            key=lambda x: abs(
                pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_utc
            ),
        )

        selected_item = pc.sign(items[0])

        # Changed: load bands with retry
        data = retry(lambda: stac_load(
            [selected_item],
            bands=["blue", "red"],
            bbox=bbox,
            resolution=30,
        ).isel(time=0))

        blue = data["blue"].astype("float")
        red = data["red"].astype("float")

        median_blue = float(blue.median(skipna=True).values)
        median_red = float(red.median(skipna=True).values)

        # Changed: treat 0 as missing
        median_blue = median_blue if median_blue != 0 else np.nan
        median_red = median_red if median_red != 0 else np.nan

        return pd.Series({
            "ls_blue": median_blue,
            "ls_red": median_red,
        })

    except Exception:
        # Changed: NEVER raise error (prevents batch crash)
        return pd.Series({
            "ls_blue": np.nan,
            "ls_red": np.nan,
        })

In [5]:
# Changed: batch extraction for extras (TRAIN)

import os
tqdm.pandas()

BATCH_SIZE = 200  # Changed
OUT_DIR = "./landsat_extra_batches_train"  # Changed
os.makedirs(OUT_DIR, exist_ok=True)  # Changed

for start in range(5000, len(base_ls), BATCH_SIZE):  # Changed
    end = min(start + BATCH_SIZE, len(base_ls))  # Changed
    out_path = f"{OUT_DIR}/ls_extra_{start}_{end}.csv"  # Changed

    if os.path.exists(out_path):
        print("Skip:", out_path)
        continue

    batch = base_ls.iloc[start:end][KEYS].copy()  # Changed
    extra = batch.progress_apply(compute_Landsat_extra, axis=1)  # Changed

    # Changed: attach merge keys
    extra[KEYS] = batch[KEYS].values  # Changed

    extra.to_csv(out_path, index=False)  # Changed
    print("Saved:", out_path)

Skip: ./landsat_extra_batches_train/ls_extra_5000_5200.csv
Skip: ./landsat_extra_batches_train/ls_extra_5200_5400.csv
Skip: ./landsat_extra_batches_train/ls_extra_5400_5600.csv
Skip: ./landsat_extra_batches_train/ls_extra_5600_5800.csv
Skip: ./landsat_extra_batches_train/ls_extra_5800_6000.csv
Skip: ./landsat_extra_batches_train/ls_extra_6000_6200.csv
Skip: ./landsat_extra_batches_train/ls_extra_6200_6400.csv
Skip: ./landsat_extra_batches_train/ls_extra_6400_6600.csv


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [09:12<00:00,  2.76s/it]


Saved: ./landsat_extra_batches_train/ls_extra_6600_6800.csv


100%|██████████| 200/200 [08:04<00:00,  2.42s/it]


...

Saved: ./landsat_extra_batches_train/ls_extra_8800_9000.csv


100%|██████████| 200/200 [10:00<00:00,  3.00s/it]


Saved: ./landsat_extra_batches_train/ls_extra_9000_9200.csv


100%|██████████| 119/119 [05:34<00:00,  2.81s/it]

Saved: ./landsat_extra_batches_train/ls_extra_9200_9319.csv


In [5]:
# Changed: merge extras into baseline (TRAIN)

import glob

files = sorted(glob.glob("./landsat_extra_batches_train/ls_extra_*.csv"))  # Changed
extra_all = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)  # Changed

# Changed: safe merge (do NOT rely on row order)
merged_train = base_ls.merge(extra_all, on=KEYS, how="left", validate="one_to_one")  # Changed

# Changed: save as a NEW file (keep original untouched)
merged_train.to_csv("./data/landsat_features_training_plus.csv", index=False)  # Changed
print("merged_train shape:", merged_train.shape)  # Changed
print("new cols:", [c for c in merged_train.columns if c.startswith("ls_")])  # Changed

merged_train shape: (9319, 13)
new cols: ['ls_blue', 'ls_red']


In [6]:
import pandas as pd
import numpy as np

orig = pd.read_csv("./data/landsat_features_training.csv")
plus = pd.read_csv("./data/landsat_features_training_plus.csv")

KEYS = ["Latitude", "Longitude", "Sample Date"]

print("===== SHAPE CHECK =====")
print("orig shape:", orig.shape)
print("plus shape:", plus.shape)

print("\n===== COLUMN CHECK =====")
print("orig columns:", orig.columns.tolist())
print("plus columns:", plus.columns.tolist())

print("\n===== DUPLICATE KEY CHECK =====")
print("orig duplicate keys:", orig.duplicated(subset=KEYS).sum())
print("plus duplicate keys:", plus.duplicated(subset=KEYS).sum())

print("\n===== KEY SET CHECK =====")
orig_keys = set(map(tuple, orig[KEYS].values))
plus_keys = set(map(tuple, plus[KEYS].values))

print("keys only in orig:", len(orig_keys - plus_keys))
print("keys only in plus:", len(plus_keys - orig_keys))

print("\n===== EXISTING COLUMN VALUE CHECK =====")
common_cols = [c for c in orig.columns if c in plus.columns and c not in KEYS]

comparison = orig[KEYS + common_cols].merge(
    plus[KEYS + common_cols],
    on=KEYS,
    suffixes=("_orig", "_plus"),
    how="inner",
    validate="one_to_one"
)

diff_summary = {}
for col in common_cols:
    same = np.isclose(
        comparison[f"{col}_orig"],
        comparison[f"{col}_plus"],
        equal_nan=True
    )
    diff_summary[col] = (~same).sum()

diff_df = pd.DataFrame.from_dict(diff_summary, orient="index", columns=["different_row_count"])
print(diff_df)

print("\n===== NEW COLUMN CHECK =====")
new_cols = [c for c in plus.columns if c not in orig.columns]
print("new columns:", new_cols)

if set(["ls_blue", "ls_red"]).issubset(new_cols):
    print("\n===== NEW COLUMN MISSING RATIO =====")
    print(plus[["ls_blue", "ls_red"]].isna().mean())

    print("\n===== NEW COLUMN SUMMARY =====")
    print(plus[["ls_blue", "ls_red"]].describe())

print("\n===== RANDOM SAMPLE CHECK =====")
sample = plus[KEYS + ["ls_blue", "ls_red"]].sample(10, random_state=42)
print(sample)

===== SHAPE CHECK =====
orig shape: (9319, 9)
plus shape: (9319, 13)

===== COLUMN CHECK =====
orig columns: ['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']
plus columns: ['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', '__date', '__key_date', 'ls_blue', 'ls_red']

===== DUPLICATE KEY CHECK =====
orig duplicate keys: 0
plus duplicate keys: 0

===== KEY SET CHECK =====
keys only in orig: 0
keys only in plus: 0

===== EXISTING COLUMN VALUE CHECK =====
        different_row_count
nir                       0
green                     0
swir16                    0
swir22                    0
NDMI                      0
MNDWI                     0

===== NEW COLUMN CHECK =====
new columns: ['__date', '__key_date', 'ls_blue', 'ls_red']

===== NEW COLUMN MISSING RATIO =====
ls_blue    0.073828
ls_red     0.073828
dtype: float64

===== NEW COLUMN SUMMARY =====
            ls_blue        ls_red
count   863

In [7]:
df = pd.read_csv("./data/landsat_features_training_plus.csv")

df = df.drop(columns=["__date", "__key_date"])

df.to_csv("./data/landsat_features_training_plus.csv", index=False)

In [1]:
# # Setup
# tqdm.pandas()

# catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=pc.sign_inplace,
#     )

# def compute_Landsat_values(row):
#     lat = row['Latitude']
#     lon = row['Longitude']
#     date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

#     # Buffer size for ~100m 
#     bbox_size = 0.005
#     # bbox_size = 0.00089831  
#     bbox = [
#         lon - bbox_size / 2,
#         lat - bbox_size / 2,
#         lon + bbox_size / 2,
#         lat + bbox_size / 2
#     ]

#     # Wider search range, we'll filter to nearest date later
#     search = catalog.search(
#         collections=["landsat-c2-l2"],
#         bbox=bbox,
#         datetime="2011-01-01/2015-12-31",
#         query={"eo:cloud_cover": {"lt": 10}},
#     )
    
#     items = search.item_collection()

#     if not items:
#         return pd.Series({
#             "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
#         })

#     try:
#         # Convert sample date to UTC
#         sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

#         # Pick the item closest to the sample date
#         items = sorted(
#             items,
#             key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
#         )
#         selected_item = pc.sign(items[0])

#         # Load required bands
#         bands_of_interest = [
#             "blue",
#             "green",
#             "red",
#             "nir08",
#             "swir16",
#             "swir22",
#         ]
#         data = stac_load(
#             [selected_item],
#             bands=bands_of_interest,
#             bbox=bbox,
#             resolution=30,   # Changed
#         ).isel(time=0)

#         blue = data["blue"].astype("float")   # Changed
#         red = data["red"].astype("float")     # Changed
#         green = data["green"].astype("float")
#         nir = data["nir08"].astype("float")
#         swir16 = data["swir16"].astype("float")
#         swir22 = data["swir22"].astype("float")
        
#         # Compute medians
#         median_blue = float(blue.median(skipna=True).values)   # Changed
#         median_red = float(red.median(skipna=True).values)     # Changed
#         median_green = float(green.median(skipna=True).values)
#         median_nir = float(nir.median(skipna=True).values)
#         median_swir16 = float(swir16.median(skipna=True).values)
#         median_swir22 = float(swir22.median(skipna=True).values)

#         # Replace 0 with NaN
#         median_blue = median_blue if median_blue != 0 else np.nan   # Changed
#         median_red = median_red if median_red != 0 else np.nan
#         median_green = median_green if median_green != 0 else np.nan
#         median_nir = median_nir if median_nir != 0 else np.nan
#         median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
#         median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
#         return pd.Series({  # Changed (missing return fixed + consistent column names)
#             "blue": median_blue,
#             "red": median_red,
#             "green": median_green,
#             "nir": median_nir,
#             "swir16": median_swir16,
#             "swir22": median_swir22,
#         })
    
#     except Exception as e:
#         return _empty_landsat_row()
    
    


### Extracting features for the training dataset

In [2]:
Water_Quality_df=pd.read_csv('./data/water_quality_training_dataset.csv')
Water_Quality_df.head()

NameError: name 'pd' is not defined

In [24]:
Water_Quality_df.shape

(9319, 6)

In [25]:
Water_Quality_df_200 = Water_Quality_df.loc[0:199]
Water_Quality_df_200.shape

(200, 6)

In [26]:
# Changed: quick single-row debug
one = compute_Landsat_values(Water_Quality_df_200.iloc[0])  # Changed
print(type(one))                                            # Changed
print(one)                                                  # Changed
print("keys:", list(one.index))                             # Changed

<class 'pandas.core.series.Series'>
blue       9774.0
red       12802.0
green     11574.0
nir       13198.0
swir16    11823.0
swir22    10939.5
dtype: float64
keys: ['blue', 'red', 'green', 'nir', 'swir16', 'swir22']


<h3>Note:</h3>
<p>The Landsat data extraction process for all 9,319 locations typically requires 7+ hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.</p>

<p>In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.</p>

<p>We have already executed the full extraction for all 9,319 locations and saved the output to <b>landsat_features_training.csv</b>, which will be used in the benchmark notebook.
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as <b>landsat_features_training.csv</b> to ensure the benchmark notebook runs smoothly.</p>

In [ ]:
# Changed: load baseline landsat (given by organizers)
base_ls = pd.read_csv("./data/landsat_features_training.csv")  # Changed

# Changed: unify keys (keep original string too)
KEYS = ["Latitude", "Longitude", "Sample Date"]  # Changed
base_ls["__date"] = pd.to_datetime(base_ls["Sample Date"], dayfirst=True, errors="coerce")  # Changed
base_ls["__key_date"] = base_ls["__date"].dt.strftime("%Y-%m-%d")  # Changed

In [ ]:
# Changed: batch extraction loop (run once)

import os
import pandas as pd

BATCH_SIZE = 200
OUT_DIR = "./landsat_batches"
os.makedirs(OUT_DIR, exist_ok=True)

N = len(Water_Quality_df)

for start in range(0, N, BATCH_SIZE):
    end = min(start + BATCH_SIZE, N)
    out_path = f"{OUT_DIR}/landsat_train_{start}_{end}.csv"

    if os.path.exists(out_path):
        print(f"Skip existing: {out_path}")
        continue

    batch_df = Water_Quality_df.iloc[start:end].copy()

    print(f"Batch {start}:{end} extracting...")
    feats = batch_df.progress_apply(compute_Landsat_values, axis=1)

    # NDMI / MNDWI (safe eps)
    eps = 1e-10
    feats["NDMI"] = (feats["nir"] - feats["swir16"]) / (feats["nir"] + feats["swir16"] + eps)
    feats["MNDWI"] = (feats["green"] - feats["swir16"]) / (feats["green"] + feats["swir16"] + eps)

    # attach keys (IMPORTANT: use batch_df, not full df)  # Changed
    feats["Latitude"] = batch_df["Latitude"].values
    feats["Longitude"] = batch_df["Longitude"].values
    feats["Sample Date"] = batch_df["Sample Date"].values

    feats.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

🚀 Running Landsat feature extraction for training data...


100%|██████████| 200/200 [14:26<00:00,  4.33s/it]


In [ ]:
# Changed: merge batch outputs into a single file

import glob
import pandas as pd

files = sorted(glob.glob("./landsat_batches/landsat_train_*.csv"))
merged = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

merged.to_csv("./data/landsat_features_training.csv", index=False)
print("Merged shape:", merged.shape)

<p><b>NDMI and MNDWI Indices:</b></p>
<p>In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:</p>
<ul>
  <li><b>NDMI (Normalized Difference Moisture Index):</b> Measures vegetation water content and surface moisture. Computed as <em>(NIR - SWIR16) / (NIR + SWIR16)</em>.</li>
  <li><b>MNDWI (Modified Normalized Difference Water Index):</b> Highlights open water features by enhancing water reflectance and suppressing built-up areas. Computed as <em>(Green - SWIR16) / (Green + SWIR16)</em>.</li>
</ul>

<p>An <b>epsilon value</b> (<em>eps = 1e-10</em>) is added in the denominators to avoid division by zero. These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.</p>


In [28]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [29]:
landsat_train_features.columns.tolist()

['blue', 'red', 'green', 'nir', 'swir16', 'swir22', 'NDMI', 'MNDWI']

In [ ]:
# landsat_train_features['Latitude'] = Water_Quality_df['Latitude']
# landsat_train_features['Longitude'] = Water_Quality_df['Longitude']
# landsat_train_features['Sample Date'] = Water_Quality_df['Sample Date']

KeyError: "['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI'] not in index"

In [ ]:
landsat_train_features.to_csv(train_features_path, index=False)

In [10]:
# Preview File
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


### Extracting features for the validation dataset

In [9]:
Validation_df=pd.read_csv('./data/submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [10]:
Validation_df.shape

(200, 6)

In [13]:
# Extract band values from Landsat for validation dataset
val_features_path = "landsat_features_validation_plus.csv"

print("🚀 Running Landsat feature extraction for validation data...")

# Landsat base bands
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)

# Landsat extra bands (blue, red)
landsat_val_extra = Validation_df.progress_apply(compute_Landsat_extra, axis=1)

# Merge base + extra
landsat_val_features = pd.concat([landsat_val_features, landsat_val_extra], axis=1)

# --------------------------------------------------
# Create indices
# --------------------------------------------------
eps = 1e-10

landsat_val_features['NDMI'] = (
    landsat_val_features['nir'] - landsat_val_features['swir16']
) / (
    landsat_val_features['nir'] + landsat_val_features['swir16'] + eps
)

landsat_val_features['MNDWI'] = (
    landsat_val_features['green'] - landsat_val_features['swir16']
) / (
    landsat_val_features['green'] + landsat_val_features['swir16'] + eps
)

# --------------------------------------------------
# Attach coordinates
# --------------------------------------------------
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']

# Final column order
landsat_val_features = landsat_val_features[
    [
        'Latitude',
        'Longitude',
        'Sample Date',
        'nir',
        'green',
        'swir16',
        'swir22',
        'NDMI',
        'MNDWI',
        'ls_blue',
        'ls_red'
    ]
]

# Save
landsat_val_features.to_csv(val_features_path, index=False)

print("✅ Validation Landsat features saved:", val_features_path)

🚀 Running Landsat feature extraction for validation data...


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [14:39<00:00,  4.40s/it]

✅ Validation Landsat features saved: landsat_features_validation_plus.csv


In [13]:
# Extract band values from Landsat for submission dataset
val_features_path = "landsat_features_validation.csv"

print("🚀 Running Landsat feature extraction for validation data...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

🚀 Running Landsat feature extraction for validation data...


100%|██████████| 200/200 [10:38<00:00,  3.19s/it]


In [ ]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'] + eps)
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [15]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [16]:
landsat_val_features.to_csv(val_features_path, index=False)

In [17]:
# Preview File
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052
